# Rebased GEVP Experiments for $\pi\pi$ Scattering Correlators

This notebook studies generalized-eigenvalue-problem (GEVP) analyses of HPC lattice scattering data.  The main goal is to compare ordinary and rebased GEVP workflows for extracting effective energies from $\pi\pi$ correlator matrices.

The workflow is organized around five ideas:

1. Load jackknife and block-double-jackknife correlator data from the HDF5 file.
2. Build several operator bases, including the full $3\times 3$ basis and two $2\times 2$ ablations.
3. Compute effective energies with and without rebasing.
4. Fit plateaus using constant fits and covariance information.
5. Run additional hyperparameter-selection tests to compare choices of $t_0$, $\Delta t$, rebasing time, rebasing separation, operator basis, and fit window.

> **Note.** This notebook assumes the project package `PySaRLAC` is importable from `../../src` and that `../../../combined_data.hdf5` is available relative to this notebook.

## 1. Imports and project setup

Load scientific Python tools, configure autoreload, and add the local `PySaRLAC` source tree to the import path.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os



# Get the absolute path of the directory containing the module
module_dir = os.path.abspath('../../src')

# Add the directory to sys.path
sys.path.insert(0, module_dir)
#for path in sys.path:
#    print(path)

import PySaRLAC as sl
import random
import math
import numpy as np
import h5py
import matplotlib as mpl
import matplotlib.pyplot as pyplot
random.seed(1234)

## 2. Operator labels

Define the mapping between integer operator indices in the HDF5 file and the physics operator names used in plots.

In [ ]:
#For reference, this is the mapping between operator index and operator name
idx_op_map = ["PiPiGnd","PiPiExc","Sigma"]
op_idx_map = dict()
for i in range(len(idx_op_map)):
    op_idx_map[idx_op_map[i]] = i

## 3. Open the HDF5 data file

Open the combined correlator file and inspect its top-level keys.

In [ ]:
f = h5py.File('../../../combined_data.hdf5','r')

f.keys()

## 4. Inspect available correlator pairs

Print the operator pairs present in the jackknife correlator data set.

In [ ]:
#These are the operator pairs in the data set
jdata = f['j_data']
con = jdata['contains']['entries']
for e in con.keys():
    op1 = con[e].attrs["first"][0]
    op2 = con[e].attrs["second"][0]
    print(op1,op2,idx_op_map[op1],idx_op_map[op2])

## 5. Build jackknife correlator matrices

Read the upper-triangular part of the $3\times3$ correlator matrix and store each entry as a `PySaRLAC` correlation function.

In [ ]:
#Lets get the jackknife data now
#we'll store just the upper triangular part of the correlator matrix
Lt=64
block_size = 8
nsample_unblocked = 741
nblock = nsample_unblocked // block_size
jdata = f['j_data']
jdata_cors = [ [None for j in range(3)] for i in range(3)] #3 ops
for i in range(3):
    for j in range(i,3):
        jdata_cors[i][j] = sl.CorrelationFunction(Lt)
        cdata = jdata['correlators']['m']["elem_%d_%d" % (i,j)]['series']
        for t in range(Lt):
            jvals = cdata["elem_%d" % t]['second'].attrs['data']
            assert len(jvals) == nblock
            jdist = sl.JackknifeDistribution(nblock)
            for s in range(nblock):
                jdist[s] = jvals[s]                        
            print(i,j,t,jdist)
            jdata_cors[i][j].setValue(t, jdist)
            jdata_cors[i][j].setCoord(t, float(t))

## 6. Build block-double-jackknife correlator matrices

Load the block-double-jackknife version of the same correlators.  These are useful for covariance estimates and cross-checking uncertainty estimates.

In [ ]:
#For block double jackknife we expect a flattened array of size nblock * ( block_size*nblock - block_size ) = 66976
nouter_samp = nblock
ninner_samp = block_size*nblock - block_size
nunrolled= nouter_samp * ninner_samp
bdjdata = f['bdj_data']
bdjdata_cors = [ [None for j in range(3)] for i in range(3)]  #3 ops
for i in range(3):    
    for j in range(i,3):
        cdata = bdjdata['correlators']['m']["elem_%d_%d" % (i,j)]['series']   
        bdjdata_cors[i][j] = sl.CorrelationFunction(Lt)
        for t in range(Lt):
            jvals = np.array(cdata["elem_%d" % t]['second']['data']['unrolled_data'])
            assert len(jvals) == nunrolled
            jdist = sl.BlockDoubleJackknifeDistribution(nsample_unblocked, block_size)
            assert jdist.size() == nouter_samp and jdist[0].size() == ninner_samp
            u = 0
            for so in range(nouter_samp):
                jdist[so].sampleVector()[:] = jvals[u:u+ninner_samp]
                u+=ninner_samp
                #for si in range(ninner_samp):
                #    jdist[so][si] = jvals[u]
                #    u+=1
            print(i,j,t,jdist[0])
            bdjdata_cors[i][j].setValue(t, jdist)
            bdjdata_cors[i][j].setCoord(t, float(t))

## 7. Define operator bases and rescalings

Construct reduced operator bases and optionally rescale operators by simple coefficients.  The main bases are full $3\times3$, no-$\sigma$, no-excited-$\pi\pi$, and a single ground-state $\pi\pi$ operator.

In [ ]:
jdata_cors_no_sigma = [[jdata_cors[0][0],jdata_cors[0][1]],[jdata_cors[0][1],jdata_cors[1][1]]]
jdata_cors_no_pipiEx = [[jdata_cors[0][0],jdata_cors[0][2]],[jdata_cors[0][2],jdata_cors[2][2]]]
jdata_cors_pipi_111 = [[jdata_cors[0][0]]]


alpha = 1
beta = 1
gamma = 1

coeffs = [alpha, beta, gamma]
#print(type(jdata_cors[1][0]))

#jdata_cors

for i in range(3):
    for j in range(i,3):
        #print(i,j)
        jdata_cors[i][j] = jdata_cors[i][j]*coeffs[i]*coeffs[j]

## 8. Jackknife GEVP output helper

`gevp_output` wraps the GEVP classes and returns an effective-energy correlation function for a chosen state, time extent, $\Delta t$, and optional rebasing parameters.

In [ ]:
def gevp_output(input_data, E_n, t_max, Dt, rebasing = False, rebased_time_slice = 1, rebased_Dt = 2):
    #time_array = np.arange(0, t_max)
    data = sl.CorrelationFunction(t_max)
    if rebasing == False:
        gevp_obj = sl.GEVP_OG(input_data)
        for i in range(0,t_max-1):
            for j in range(i+1, t_max):
                if j-i == Dt:
                    data.setValue(j, gevp_obj.run(i,j)[E_n])
                    data.setCoord(j, float(j))
        return  data
    if rebasing == True:
        gevp_obj = sl.GEVP(input_data)
        for i in range(0,t_max-1):
            for j in range(i+1, t_max):
                if j-i == Dt:
                    data.setValue(j, gevp_obj.run(i,j, rebased_time_slice, rebased_time_slice + 
                                                  rebased_Dt)[E_n])
                    data.setCoord(j, float(j))
        return data

## 9. Block-double-jackknife GEVP output helper

`gevp_output2` mirrors `gevp_output`, but uses block-double-jackknife distributions.

In [ ]:
def gevp_output2(input_data, E_n, t_max, Dt, rebasing = False, rebased_time_slice = 1, rebased_Dt = 2):
    #time_array = np.arange(0, t_max)
    data = sl.CorrelationFunction(t_max)
    if rebasing == False:
        gevp_obj = sl.GEVP_bd(input_data)
        for i in range(0,t_max-1):
            for j in range(i+1, t_max):
                if j-i == Dt:
                    data.setValue(j, gevp_obj.run(i,j, rebased = False)[E_n])
                    data.setCoord(j, float(j))
        return  data
    if rebasing == True:
        gevp_obj = sl.GEVP_bd(input_data)
        for i in range(0,t_max-1):
            for j in range(i+1, t_max):
                if j-i == Dt:
                    data.setValue(j, gevp_obj.run(i,j, rebased_time_slice, rebased_time_slice + 
                                                  rebased_Dt)[E_n])
                    data.setCoord(j, float(j))
        return data

## 10. First rebased $2\times2$ test: remove $\sigma$

Compare the rebased no-$\sigma$ operator basis against the corresponding non-rebased analysis.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset

fig, ax = plt.subplots()

#gevp_without_sigma = gevp_output(jdata_cors_no_sigma, 0, 15, 3)

gevp_without_sigma_rebased = gevp_output(jdata_cors_no_sigma, 0, 15, 3, rebasing=True)


#ax.errorbar(*gevp_without_sigma.plotInputs(), label = r"Removed $\sigma$ op", marker="o", capsize=5, linestyle='')
ax.errorbar(*gevp_without_sigma_rebased.plotInputs(), label = r"Removed $\sigma$ op and Rebased", marker="o", capsize=5, linestyle='')

x1, x2, y1, y2 = 3, 9, 0.35, 0.4
axins = zoomed_inset_axes(ax, zoom=2, loc='upper right')
#axins.errorbar(gevp_without_sigma.plotInputs()[0]+0.1, *gevp_without_sigma.plotInputs()[1:], marker="o", capsize=5, linestyle='')
axins.errorbar(*gevp_without_sigma_rebased.plotInputs(), marker="o", capsize=5, linestyle='')
axins.set_xlim(x1, x2)
axins.set_ylim(y1, y2)

# Hide ticks inside inset
axins.set_xticks([])
axins.set_yticks([])

# Draw lines between inset and zoomed area
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")

ax.set_title(r"Ground state with $\sigma$ operator removed")
ax.set_ylabel("Energy")
ax.set_xlabel(r"$t$")
ax.set_ylim(0.1, 0.6)
ax.legend(loc = "lower center")

## 11. Full $3\times3$ jackknife vs. block-double-jackknife comparison

Compare the rebased effective energies obtained from standard jackknife and block-double-jackknife data.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset

fig, ax = plt.subplots()

print(type(bdjdata_cors[0][0]))

gevp_without_sigma_rebased2 = gevp_output2(bdjdata_cors, 0, 15, 2, rebasing=True, rebased_Dt= 4, rebased_time_slice=2)

gevp_without_sigma_rebased = gevp_output(jdata_cors, 0, 15, 2, rebasing=True, rebased_Dt= 4, rebased_time_slice=2)

ax.errorbar(*gevp_without_sigma_rebased.plotInputs()[:-1], label = r"3 by 3 GEVP Rebased", marker="o", capsize=5, linestyle='')

x1, x2, y1, y2 = 3, 9, 0.35, 0.4
axins = zoomed_inset_axes(ax, zoom=2, loc='upper right')
axins.errorbar(*gevp_without_sigma_rebased.plotInputs(), marker="o", capsize=5, linestyle='')
axins.set_xlim(x1, x2)
axins.set_ylim(y1, y2)

# Hide ticks inside inset
axins.set_xticks([])
axins.set_yticks([])

# Draw lines between inset and zoomed area
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")

ax.set_title(r"Ground state with $\sigma$ operator removed")
ax.set_ylabel("Energy")
ax.set_xlabel(r"$t$")
ax.set_ylim(0.1, 0.6)
ax.legend(loc = "lower center")

## 12. Constant plateau fit

Fit a selected time range to a constant effective energy and compute $\chi^2/\mathrm{dof}$.

In [ ]:
t_range = (4,9)

N = gevp_without_sigma_rebased.value(2).size()

fitfunc = sl.FitConstant()

fitter = sl.Fitter(fitfunc)

dd_test = gevp_without_sigma_rebased2.sliceRange(*t_range)

fitter.generateCovarianceMatrix(dd_test)

jack_inrange = gevp_without_sigma_rebased.sliceRange(*t_range)

params = [sl.JackknifeDistribution(N, 0.38)]
chisq, dof = fitter.fit(params, jack_inrange)
print(params[0], chisq/float(dof))

#Compare results with frozen and unfrozen. Stat. result for unfrozen case should be slightly higher
#Look at the cov matrix itself. Substract on sample by sample basis. Plot in linear format all samples.
#look at tmin dependence, tabulate all result, and plots for excited state contamination


## 13. Plot the plateau fit

Visualize the fitted energy band against the selected effective-energy data.

In [ ]:
fit_y = sl.evaluateFitFunc(fitfunc,jack_inrange,params)
plot_data = jack_inrange.plotInputs()
plot_result = fit_y.plotInputs()
#print(rf"E_{energy_state}:  {plot_result[1][0]} +/- {plot_data[2][0]}")

print(jack_inrange)
pyplot.errorbar(plot_data[0],plot_data[1],yerr=plot_data[2])
pyplot.fill_between(plot_result[0],plot_result[1]-plot_result[2],plot_result[1]+plot_result[2])
plt.ylabel(r"$E_0$")
plt.savefig("gevp_lattice_1.svg", format="svg", bbox_inches="tight")

## 14. Print fitted energy estimate

Report the central value and uncertainty from the constant fit.

In [ ]:
print(plot_result[1][0], "\pm", plot_result[2][0])

In [ ]:
gevp_without_sigma_rebased2.value(3)[0]

## 15. Operator-basis ablations

Compare the no-$\pi\pi_{\rm exc}$ basis, its rebased version, and the single-operator $\pi\pi$ analysis.

In [ ]:
gevp_without_pipiEx = gevp_output(jdata_cors_no_pipiEx, 0, 15, 3)

gevp_without_pipiEx_rebased = gevp_output(jdata_cors_no_pipiEx, 0, 15, 3, rebasing=True)

gevp_without_pipi_111 = gevp_output(jdata_cors_pipi_111, 0, 15, 3)

fig, ax = plt.subplots()


ax.errorbar(*gevp_without_pipiEx.plotInputs(), label = r"Removed PiPiEx op", marker="o", capsize=5, linestyle='')
ax.errorbar(*gevp_without_pipiEx_rebased.plotInputs(), label = r"Removed PiPiEx op and Rebased", marker="o", capsize=5, linestyle='')
ax.errorbar(*gevp_without_pipi_111.plotInputs(), label = r"Only PiPi", marker="o", capsize=5, linestyle='')

x1, x2, y1, y2 = 3, 9, 0.32, 0.39
axins = zoomed_inset_axes(ax, zoom=3, loc='upper right')
axins.errorbar(gevp_without_pipiEx.plotInputs()[0]+0.1, *gevp_without_pipiEx.plotInputs()[1:], marker="o", capsize=5, linestyle='')
axins.errorbar(*gevp_without_pipiEx_rebased.plotInputs(), marker="o", capsize=5, linestyle='')
axins.errorbar(gevp_without_pipi_111.plotInputs()[0]+0.2, *gevp_without_pipi_111.plotInputs()[1:], label = r"Only PiPi", marker="o", capsize=5, linestyle='')
axins.set_xlim(x1, x2)
axins.set_ylim(y1, y2)
#axins.set_yticks()
# Hide ticks inside inset
axins.set_xticks([])
axins.set_yticks([0.34,0.35, 0.36, 0.37])

# Draw lines between inset and zoomed area
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")

ax.set_title(r"Ground state with PiPiEx operator removed")
ax.set_ylabel("Energy")
ax.set_xlabel(r"$t$")
ax.set_ylim(0.1, 0.6)
ax.legend(loc = "lower center")

## 16. Rebased basis comparison

Compare the two rebased $2\times2$ bases directly.

In [ ]:
fig, ax = plt.subplots()


ax.errorbar(*gevp_without_sigma_rebased.plotInputs(), label = r"Removed $\sigma$ op and Rebased", marker="o", capsize=5, linestyle='')
ax.errorbar(*gevp_without_pipiEx_rebased.plotInputs(), label = r"Removed PiPiEx op and Rebased", marker="o", capsize=5, linestyle='')

x1, x2, y1, y2 = 3, 9, 0.32, 0.4
axins = zoomed_inset_axes(ax, zoom=2, loc='upper right')
axins.errorbar(*gevp_without_sigma_rebased.plotInputs(), marker="o", capsize=5, linestyle='')
axins.errorbar(*gevp_without_pipiEx_rebased.plotInputs(), marker="o", capsize=5, linestyle='')
axins.set_xlim(x1, x2)
axins.set_ylim(y1, y2)

# Hide ticks inside inset
axins.set_xticks([])
axins.set_yticks([])

# Draw lines between inset and zoomed area
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")

ax.set_title(r"Comparison of Ground state wrt to time when PiPiEx  and $\sigma $operator removed")
ax.set_ylabel("Energy")
ax.set_xlabel(r"$t$")
ax.set_ylim(0.1, 0.6)
ax.legend(loc = "lower center")

## 17. Ground-state full-basis comparison

Compare the full $3\times3$ GEVP against a rebased $2\times2$ projection for the ground state.

In [ ]:
gevp_3by3 = gevp_output(jdata_cors, 0, 15, 3)
print("Rebasing")
gevp_3by3_rebased = gevp_output(jdata_cors, 0, 15, 3, rebasing=True)


fig, ax = plt.subplots()


ax.errorbar(*gevp_3by3.plotInputs(), label = r"3by3", marker="o", capsize=5, linestyle='')
ax.errorbar(gevp_3by3_rebased.plotInputs()[0]+0.1, *gevp_3by3_rebased.plotInputs()[1:], label = r"2by2 with rebasing", marker="o", capsize=5, linestyle='')
ax.legend()
ax.set_ylim(-0.3, 0.4)

#make a protection code, determine what values where I get error, disregard these point in jackknife, label these points (log(-#))


## 18. Excited-state full-basis comparison

Repeat the full-vs-rebased comparison for the first excited state.

In [ ]:
gevp_3by3_Ex = gevp_output(jdata_cors, 1, 15, 3)
print("Rebasing")
gevp_3by3_rebased_Ex = gevp_output(jdata_cors, 1, 15, 3, rebasing=True)


fig, ax = plt.subplots()


ax.errorbar(*gevp_3by3_Ex.plotInputs(), label = r"3by3 excited state", marker="o", capsize=5, linestyle='')
ax.errorbar(gevp_3by3_rebased_Ex.plotInputs()[0]+0.1, *gevp_3by3_rebased_Ex.plotInputs()[1:], label = r"2by2 with rebasing excited state", marker="o", capsize=5, linestyle='')
ax.legend()
ax.set_ylim(0, 1)


#if gevp fails then discard entire time-slice
#perform fits (Change rabased params st the orange curve error bar are similar to blue error bars)
#Write results up with all data thus far with various fits (similar to small research paper) ---> draw conclusions (prepare for monday meeting)
#Compare with 2020 results, use same t_min as paper (t_min=6), do 3by3 and 2by2 with rebasing

## 19. Existing rebasing scans

Scan rebasing choices manually by varying the rebasing time slice and rebasing separation.

In [ ]:
N_rebased_slices = 10

rebased_GEVP_for_variable_timeslice =  np.empty(N_rebased_slices, dtype=sl.CorrelationFunction)

rebased_t0 = 1

for i in range(N_rebased_slices):
    
    rebased_GEVP_for_variable_timeslice[i] = gevp_output(jdata_cors, 1, 15, 3, rebased_time_slice = i + rebased_t0, rebasing = True)
    
fig, ax = plt.subplots()

for j in range(N_rebased_slices):
    ax.errorbar(*rebased_GEVP_for_variable_timeslice[j].plotInputs(), label = rf"Rebased $t_0$: {j + rebased_t0}", marker="o", capsize=5, linestyle='')
    
ax.legend()
ax.set_ylim(0.51, 0.68)

#create data without any noise, which has 5 states. 4 and 5 state used to mimic contamination
#Step 1: Create data set with only 3 states
#Step 2: Introduce Noise (Gaussian noise), amplitude of noise can increase as function of time
#Step 3: Called noise eps(t), data * (1+eps(t)), Create raw data by regenerating noise 100s, len(dataset)=100
#Step 4: Perform Jacknife, use code to analyze dataset, we should not have any systematic error, GEVP should give energy value consistent with real value (assigned) with some stat error
#Step 5: Use real value for state op coupling and the energy and vary the amplitude of eps (gradually increase error) so we should have log(-ve)
#Step 6: Introduce the 4th state. Figure out what is the coupling between the 4th state and 3 operator, vary the coupling (all large, coupling to ground large adn sig, piex small)
#First make sure 1-3 is completely align with expectations, then Move to 6
#Check if you gradual increase  error GEVP will fail then rebasing should extend GEVP then fail
#Reminder to implement protector code in GEVP object, jackknife method for resampling which help squeeeze data to central value, extra factor of (N-1) if wedont introduce this factor
#jckknife samples under estimate the error but for log(-ve) it introduce major error, more breakdown likely in bootstrap method

In [ ]:
N_rebased_Dt = 10

rebased_GEVP_for_variable_Dt=  np.empty(N_rebased_slices, dtype=sl.CorrelationFunction)

rebased_Dt0 = 1

for i in range(N_rebased_slices):
    
    rebased_GEVP_for_variable_Dt[i] = gevp_output(jdata_cors, 1, 15, 3, rebased_Dt = i + rebased_Dt0, rebased_time_slice = 3, rebasing = True)
    
fig, ax = plt.subplots()

for j in range(N_rebased_Dt):
    ax.errorbar(*rebased_GEVP_for_variable_Dt[j].plotInputs(), label = rf"Rebased $Dt$: {j + rebased_Dt0}", marker="o", capsize=5, linestyle='')
    
ax.legend()
ax.set_ylim(0.51, 0.68)

#jackknife diff between Dt=1

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(*rebased_GEVP_for_variable_Dt[0].plotInputs(), label = rf"Rebased $Dt$: {0 + rebased_Dt0}", marker="o", capsize=5, linestyle='')
ax.errorbar(*rebased_GEVP_for_variable_timeslice[1].plotInputs(x_offset = 0.1), label = rf"Rebased $t_0$: {1 + rebased_t0}", marker="o", capsize=5, linestyle='')
ax.legend()
ax.set_ylim(0.51, 0.68)


## 20. Scratch fitting setup

This final legacy cell appears to prepare correlation functions for later fitting.  It depends on variables such as `Dt`, `j_dists`, and `j_dists2` that are not defined earlier in this cleaned notebook, so treat it as scratch space unless those variables are created upstream.

In [ ]:
# Legacy scratch cell: only run if these external variables have been defined.
if all(name in globals() for name in ["Dt", "j_dists", "j_dists2"]):
    jack_correlation = sl.CorrelationFunction(Lt)
    dd_jack = sl.CorrelationFunction(Lt)
    for t0 in range(0, Lt):
        t = t0 + Dt
        jack_correlation.setCoord(t0, float(t))
        jack_correlation.setValue(t0, j_dists[t0])
        dd_jack.setCoord(t0, float(t0))
        dd_jack.setValue(t0, j_dists2[t0])
else:
    print("Skipping legacy scratch fitting setup because Dt, j_dists, or j_dists2 is undefined.")

## 21. Additional hyperparameter-selection tests

The cells below add a compact scoring framework for choosing GEVP/rebasing hyperparameters.  The point is not to declare one number “correct,” but to identify settings that are simultaneously:

- stable under neighboring fit windows,
- statistically precise,
- not obviously overfitting noisy timeslices,
- robust to changing the operator basis,
- robust to changing the rebasing time and rebasing separation.

Each test returns a `pandas.DataFrame` sorted by a simple diagnostic score.  Smaller scores are better.  The score combines the fitted statistical error, $\chi^2/\mathrm{dof}$ when available, and a penalty for failed or non-finite fits.

In [ ]:
import itertools
import pandas as pd
import numpy as np


def _safe_plot_arrays(corr):
    """Return x, y, yerr arrays from a PySaRLAC CorrelationFunction."""
    x, y, yerr = corr.plotInputs()[:3]
    return np.asarray(x, dtype=float), np.asarray(y, dtype=float), np.asarray(yerr, dtype=float)


def _safe_constant_fit(corr, fit_range, guess=0.4):
    """
    Fit a CorrelationFunction to a constant over fit_range=(t_min,t_max).

    Returns a dictionary with fitted energy, error, chi2/dof, and a finite-data fraction.
    The function is intentionally defensive so that bad hyperparameters produce rows with
    `ok=False` rather than stopping the whole scan.
    """
    try:
        sliced = corr.sliceRange(*fit_range)
        x, y, yerr = _safe_plot_arrays(sliced)
        finite = np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)
        finite_fraction = float(np.mean(finite)) if len(y) else 0.0
        if finite.sum() < 2:
            return dict(ok=False, E=np.nan, dE=np.nan, chi2_dof=np.nan,
                        finite_fraction=finite_fraction, reason="too few finite points")

        fitfunc = sl.FitConstant()
        fitter = sl.Fitter(fitfunc)

        # Use the distribution attached to the first finite timeslice to determine sample size.
        sample_dist = corr.value(int(x[finite][0]))
        params = [sl.JackknifeDistribution(sample_dist.size(), guess)]
        chisq, dof = fitter.fit(params, sliced)

        # Extract the fit value through PySaRLAC's plotting interface.  This matches
        # the earlier manual fit/plot cells and avoids relying on distribution method names.
        fit_y = sl.evaluateFitFunc(fitfunc, sliced, params)
        _, fit_mean, fit_err = fit_y.plotInputs()[:3]
        E = float(np.asarray(fit_mean, dtype=float)[0])
        dE = float(np.asarray(fit_err, dtype=float)[0])
        chi2_dof = float(chisq) / float(dof) if dof else np.nan
        return dict(ok=True, E=E, dE=dE, chi2_dof=chi2_dof,
                    finite_fraction=finite_fraction, reason="")
    except Exception as err:
        return dict(ok=False, E=np.nan, dE=np.nan, chi2_dof=np.nan,
                    finite_fraction=0.0, reason=repr(err))


def _score_fit(row, target_chi2=1.0):
    """Simple scalar diagnostic score: smaller is better."""
    if not row.get("ok", False):
        return np.inf
    dE = row.get("dE", np.nan)
    chi2 = row.get("chi2_dof", np.nan)
    finite_penalty = 1.0 - row.get("finite_fraction", 0.0)
    chi2_penalty = abs(chi2 - target_chi2) if np.isfinite(chi2) else 1.0
    err_term = dE if np.isfinite(dE) else 1.0
    return float(err_term + 0.05 * chi2_penalty + finite_penalty)


def _make_gevp(input_data, E_n, t_max, Dt, rebasing=False, rebased_time_slice=1, rebased_Dt=2):
    """Centralized wrapper for all scans."""
    return gevp_output(
        input_data=input_data,
        E_n=E_n,
        t_max=t_max,
        Dt=Dt,
        rebasing=rebasing,
        rebased_time_slice=rebased_time_slice,
        rebased_Dt=rebased_Dt,
    )


def summarize_top(df, n=10):
    """Display the best rows using the diagnostic score."""
    return df.sort_values("score", ascending=True).head(n).reset_index(drop=True)

In [ ]:
# Test 1: scan the ordinary GEVP time separation Dt and plateau fit window.
# Question: which (Dt, fit window) gives a stable, precise plateau before rebasing?

def scan_plain_gevp_dt_and_fit_windows(input_data, E_n=0, t_max=15,
                                       Dt_values=range(1, 6),
                                       fit_starts=range(3, 8),
                                       fit_lengths=(3, 4, 5),
                                       guess=0.4):
    rows = []
    for Dt in Dt_values:
        try:
            corr = _make_gevp(input_data, E_n=E_n, t_max=t_max, Dt=Dt, rebasing=False)
        except Exception as err:
            rows.append(dict(test="plain_dt_fit", Dt=Dt, fit_range=None, ok=False,
                             E=np.nan, dE=np.nan, chi2_dof=np.nan, finite_fraction=0.0,
                             reason=repr(err), score=np.inf))
            continue
        for start, length in itertools.product(fit_starts, fit_lengths):
            fit_range = (start, start + length)
            row = _safe_constant_fit(corr, fit_range, guess=guess)
            row.update(test="plain_dt_fit", Dt=Dt, fit_range=fit_range)
            row["score"] = _score_fit(row)
            rows.append(row)
    return pd.DataFrame(rows)

plain_dt_scan = scan_plain_gevp_dt_and_fit_windows(jdata_cors, E_n=0)
summarize_top(plain_dt_scan)

In [ ]:
# Test 2: scan rebasing time slice and rebasing Dt at fixed analysis Dt.
# Question: where should the basis be rebased?

def scan_rebasing_grid(input_data, E_n=0, t_max=15, analysis_Dt=3,
                       rebased_time_slices=range(1, 8),
                       rebased_Dt_values=range(1, 7),
                       fit_range=(4, 9),
                       guess=0.4):
    rows = []
    for tr, dtr in itertools.product(rebased_time_slices, rebased_Dt_values):
        try:
            corr = _make_gevp(input_data, E_n=E_n, t_max=t_max, Dt=analysis_Dt,
                              rebasing=True, rebased_time_slice=tr, rebased_Dt=dtr)
            row = _safe_constant_fit(corr, fit_range, guess=guess)
        except Exception as err:
            row = dict(ok=False, E=np.nan, dE=np.nan, chi2_dof=np.nan,
                       finite_fraction=0.0, reason=repr(err))
        row.update(test="rebasing_grid", analysis_Dt=analysis_Dt,
                   rebased_time_slice=tr, rebased_Dt=dtr, fit_range=fit_range)
        row["score"] = _score_fit(row)
        rows.append(row)
    return pd.DataFrame(rows)

rebasing_grid_scan = scan_rebasing_grid(jdata_cors, E_n=0)
summarize_top(rebasing_grid_scan)

In [ ]:
# Test 3: rolling-window plateau stability for each rebasing choice.
# Question: does the fitted energy remain stable when the fit window is shifted?

def scan_plateau_stability(input_data, E_n=0, t_max=15, analysis_Dt=3,
                           rebased_time_slices=range(1, 7),
                           rebased_Dt_values=range(1, 6),
                           window_length=4,
                           window_starts=range(3, 8),
                           guess=0.4):
    rows = []
    for tr, dtr in itertools.product(rebased_time_slices, rebased_Dt_values):
        window_fits = []
        try:
            corr = _make_gevp(input_data, E_n=E_n, t_max=t_max, Dt=analysis_Dt,
                              rebasing=True, rebased_time_slice=tr, rebased_Dt=dtr)
            for start in window_starts:
                fit_range = (start, start + window_length)
                fit = _safe_constant_fit(corr, fit_range, guess=guess)
                fit.update(fit_range=fit_range)
                window_fits.append(fit)
            good = [f for f in window_fits if f["ok"] and np.isfinite(f["E"])]
            energies = np.asarray([f["E"] for f in good])
            mean_err = np.nanmean([f["dE"] for f in good]) if good else np.nan
            mean_chi2 = np.nanmean([f["chi2_dof"] for f in good]) if good else np.nan
            stability = float(np.nanstd(energies)) if len(energies) > 1 else np.inf
            ok = len(good) >= 2
            row = dict(ok=ok, E=float(np.nanmean(energies)) if ok else np.nan,
                       dE=mean_err, chi2_dof=mean_chi2,
                       finite_fraction=len(good) / max(len(window_fits), 1),
                       plateau_std=stability, reason="")
        except Exception as err:
            row = dict(ok=False, E=np.nan, dE=np.nan, chi2_dof=np.nan,
                       finite_fraction=0.0, plateau_std=np.inf, reason=repr(err))
        row.update(test="plateau_stability", analysis_Dt=analysis_Dt,
                   rebased_time_slice=tr, rebased_Dt=dtr,
                   window_length=window_length)
        row["score"] = _score_fit(row) + row["plateau_std"]
        rows.append(row)
    return pd.DataFrame(rows)

plateau_stability_scan = scan_plateau_stability(jdata_cors, E_n=0)
summarize_top(plateau_stability_scan)

In [ ]:
# Test 4: operator-basis ablation scan.
# Question: is the preferred energy robust to dropping sigma or the excited pi-pi operator?

def scan_operator_bases(E_n=0, t_max=15, analysis_Dt=3,
                        rebased_time_slice=2, rebased_Dt=4,
                        fit_range=(4, 9), guess=0.4):
    bases = {
        "full_3x3": jdata_cors,
        "no_sigma_2x2": jdata_cors_no_sigma,
        "no_pipiExc_2x2": jdata_cors_no_pipiEx,
        "pipi_ground_1x1": jdata_cors_pipi_111,
    }
    rows = []
    for basis_name, input_data in bases.items():
        for rebasing in (False, True):
            # A 1x1 basis cannot meaningfully be reduced by rebasing.
            if basis_name == "pipi_ground_1x1" and rebasing:
                continue
            try:
                corr = _make_gevp(input_data, E_n=0 if basis_name == "pipi_ground_1x1" else E_n,
                                  t_max=t_max, Dt=analysis_Dt, rebasing=rebasing,
                                  rebased_time_slice=rebased_time_slice, rebased_Dt=rebased_Dt)
                row = _safe_constant_fit(corr, fit_range, guess=guess)
            except Exception as err:
                row = dict(ok=False, E=np.nan, dE=np.nan, chi2_dof=np.nan,
                           finite_fraction=0.0, reason=repr(err))
            row.update(test="operator_basis", basis=basis_name, rebasing=rebasing,
                       analysis_Dt=analysis_Dt, rebased_time_slice=rebased_time_slice,
                       rebased_Dt=rebased_Dt, fit_range=fit_range)
            row["score"] = _score_fit(row)
            rows.append(row)
    df = pd.DataFrame(rows)
    if df["ok"].any():
        reference = df.loc[df["ok"], "E"].median()
        df["basis_shift_from_median"] = np.abs(df["E"] - reference)
        df["score"] = df["score"] + df["basis_shift_from_median"].fillna(1.0)
    return df

basis_scan = scan_operator_bases(E_n=0)
summarize_top(basis_scan)

In [ ]:
# Test 5: jackknife vs block-double-jackknife consistency.
# Question: do the preferred hyperparameters remain reasonable when the uncertainty estimator changes?

def scan_jackknife_vs_bdj(E_n=0, t_max=15, analysis_Dt=2,
                          rebased_time_slice=2, rebased_Dt=4,
                          fit_range=(4, 9), guess=0.4):
    rows = []
    configs = [
        ("jackknife", lambda: gevp_output(jdata_cors, E_n, t_max, analysis_Dt,
                                          rebasing=True,
                                          rebased_time_slice=rebased_time_slice,
                                          rebased_Dt=rebased_Dt)),
        ("block_double_jackknife", lambda: gevp_output2(bdjdata_cors, E_n, t_max, analysis_Dt,
                                                        rebasing=True,
                                                        rebased_time_slice=rebased_time_slice,
                                                        rebased_Dt=rebased_Dt)),
    ]
    for estimator, factory in configs:
        try:
            corr = factory()
            row = _safe_constant_fit(corr, fit_range, guess=guess)
        except Exception as err:
            row = dict(ok=False, E=np.nan, dE=np.nan, chi2_dof=np.nan,
                       finite_fraction=0.0, reason=repr(err))
        row.update(test="uncertainty_estimator", estimator=estimator,
                   analysis_Dt=analysis_Dt, rebased_time_slice=rebased_time_slice,
                   rebased_Dt=rebased_Dt, fit_range=fit_range)
        row["score"] = _score_fit(row)
        rows.append(row)
    df = pd.DataFrame(rows)
    if df["ok"].sum() == 2:
        delta = abs(df.loc[df["estimator"] == "jackknife", "E"].iloc[0]
                    - df.loc[df["estimator"] == "block_double_jackknife", "E"].iloc[0])
        df["jackknife_bdj_energy_difference"] = delta
        df["score"] = df["score"] + delta
    return df

uncertainty_scan = scan_jackknife_vs_bdj(E_n=0)
uncertainty_scan.sort_values("estimator")

In [ ]:
# Test 6: combined recommendation table.
# Question: among a moderate grid, which hyperparameters pass several diagnostics at once?

def combined_hyperparameter_recommendation(input_data, E_n=0, t_max=15,
                                           analysis_Dt_values=(2, 3, 4),
                                           rebased_time_slices=range(1, 6),
                                           rebased_Dt_values=range(2, 6),
                                           fit_windows=((3, 7), (4, 8), (4, 9), (5, 9)),
                                           guess=0.4):
    rows = []
    for analysis_Dt, tr, dtr, fit_range in itertools.product(
        analysis_Dt_values, rebased_time_slices, rebased_Dt_values, fit_windows
    ):
        try:
            corr = _make_gevp(input_data, E_n=E_n, t_max=t_max, Dt=analysis_Dt,
                              rebasing=True, rebased_time_slice=tr, rebased_Dt=dtr)
            row = _safe_constant_fit(corr, fit_range, guess=guess)
        except Exception as err:
            row = dict(ok=False, E=np.nan, dE=np.nan, chi2_dof=np.nan,
                       finite_fraction=0.0, reason=repr(err))
        row.update(test="combined", analysis_Dt=analysis_Dt,
                   rebased_time_slice=tr, rebased_Dt=dtr, fit_range=fit_range)
        row["score"] = _score_fit(row)
        rows.append(row)
    df = pd.DataFrame(rows)

    # Add a robustness penalty: settings with energies far from the median of successful fits are down-ranked.
    if df["ok"].any():
        median_E = df.loc[df["ok"], "E"].median()
        df["energy_shift_from_successful_median"] = np.abs(df["E"] - median_E)
        df["score"] = df["score"] + df["energy_shift_from_successful_median"].fillna(1.0)
    return df

recommendation_scan = combined_hyperparameter_recommendation(jdata_cors, E_n=0)
best_hyperparams = summarize_top(recommendation_scan, n=15)
best_hyperparams

In [ ]:
# Optional visualization: plot the best recommended rebased correlator and its selected fit window.
# Run this after `best_hyperparams` has been computed.

best = best_hyperparams.iloc[0]
fit_range = tuple(best["fit_range"])
recommended_corr = _make_gevp(
    jdata_cors,
    E_n=0,
    t_max=15,
    Dt=int(best["analysis_Dt"]),
    rebasing=True,
    rebased_time_slice=int(best["rebased_time_slice"]),
    rebased_Dt=int(best["rebased_Dt"]),
)

fit = _safe_constant_fit(recommended_corr, fit_range, guess=float(best["E"]))
x, y, yerr = _safe_plot_arrays(recommended_corr)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(x, y, yerr=yerr, marker="o", capsize=5, linestyle="", label="Rebased GEVP")
ax.axvspan(fit_range[0], fit_range[1], alpha=0.15, label=f"fit window {fit_range}")
if fit["ok"]:
    ax.axhline(fit["E"], linestyle="--", label=rf"$E={fit['E']:.4f}$")
ax.set_xlabel(r"$t$")
ax.set_ylabel(r"$E_{\mathrm{eff}}(t)$")
ax.set_title("Best-scored rebased GEVP hyperparameter setting")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("Recommended hyperparameters:")
print(best[["analysis_Dt", "rebased_time_slice", "rebased_Dt", "fit_range", "E", "dE", "chi2_dof", "score"]])